# Creating dataset for training YOLO model with RTMO-X

### Dataset construction with RTMO-X

In [23]:
# DEPENDENCIES

import glob
import time
import cv2
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)
print(torch.__version__)

from pathlib import Path
from mmpose.apis import init_model, inference_topdown
from mmpose.visualization import PoseLocalVisualizer
from mmdet.apis import init_detector, inference_detector
from enum import Enum
from typing import List

cuda
2.1.2+cu118


In [24]:
# SETUP

KEYPOINT_IDS = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

CONFIG_ROOT_PATH = "D:\\Magistrska\\mmpose\\configs\\"
CHECKPOINT_ROOT_PATH = "D:\\Magistrska\\mmpose\\checkpoints\\"


def get_model_cfg_and_checkpoint():
    cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\rtmo\\coco\\rtmo-x_8xb256-700e_coco-384x288.py"
    checkpoint_path = f"{CHECKPOINT_ROOT_PATH}rtmo\\coco\\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth"

    return cfg_path, checkpoint_path


def get_detector_cfg_and_checkpoint():
    cfg_path = f"{CONFIG_ROOT_PATH}body_2d_keypoint\\yoloxpose\\coco\\yoloxpose_l_8xb32-300e_coco-640.py"
    checkpoint_path = f"{CHECKPOINT_ROOT_PATH}yolox\\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth"

    return cfg_path, checkpoint_path

In [25]:
# KEYPOINT UTIL


class Keypoint:
    def __init__(self, id, x, y):
        self.id = id
        self.pixelPosition = (x, y)


def processKeypoints(keypoints: list, width: int, height: int) -> List[Keypoint]:
    final_keypoints: list[Keypoint] = []

    for i, keypoint in enumerate(keypoints):
        keypoint_id = KEYPOINT_IDS[i]
        x = float(keypoint[0]) / width
        y = float(keypoint[1]) / height
        final_keypoints.append(Keypoint(keypoint_id, x, y))

    return final_keypoints

def createTxtFileFromKeypoints(img_name: str, width: int, height: int, keypoints: List[Keypoint], output_dir: str): #
    txt_filename = f"{Path(img_name).stem}.txt"
    output_path = Path(output_dir) / txt_filename

    person_class = 0

    x_min = min(k.pixelPosition[0] for k in keypoints)
    x_max = max(k.pixelPosition[0] for k in keypoints)
    y_min = min(k.pixelPosition[1] for k in keypoints)
    y_max = max(k.pixelPosition[1] for k in keypoints)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    bbox_width = x_max - x_min
    bbox_height = y_max - y_min

    keypoints_string = ''.join([f"{kp.pixelPosition[0]:.6f} {kp.pixelPosition[1]:.6f} " for kp in keypoints]).strip()

    out_string = f"{person_class} {x_center:.6f} {y_center:.6f} {bbox_width} {bbox_height} {keypoints_string}"

    with open(output_path, 'w') as f:
        f.write(out_string)

In [26]:
# CONSTANTS

DATASET_DIR = r'D:\Magistrska\blindoff-dataset\selected_images'
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.tif", "*.tiff")

In [28]:
# RECOGNITION ON IMAGES

model_cfg_path, model_checkpoint_path = get_model_cfg_and_checkpoint()
detector_cfg_path, detector_checkpoint_path = get_detector_cfg_and_checkpoint()

pose_model = init_model(model_cfg_path, model_checkpoint_path, device=device)
det_model = init_detector(detector_cfg_path, detector_checkpoint_path, device=device)

input_dir = Path(DATASET_DIR)

image_paths = []
for ext in IMG_EXTS:
    image_paths.extend(glob.glob(str(input_dir / ext)))
image_paths = sorted(image_paths)

total = len(image_paths)

for idx, img_path in enumerate(image_paths, start=1):
    print(f"Processing image ({idx}/{total})")

    frame = cv2.imread(img_path)
    if frame is None:
        print(f"[WARN] Could not read image: {img_path}")
        continue

    # --------- BBoxes ---------
    det_result = inference_detector(det_model, frame)
    pred_instance = det_result.pred_instances
    person_bboxes = pred_instance.bboxes[pred_instance.labels == 0]
    scores = pred_instance.scores[pred_instance.labels == 0]

    if len(person_bboxes) > 0:
        best_idx = scores.argmax().item()
        bboxes = person_bboxes[best_idx].reshape(1, -1)
    else:
        bboxes = []

    # --------- Pose inference ---------
    model_pose_results = inference_topdown(pose_model, frame, bboxes)

    # --------- Visualization ---------
    vis_frame = frame.copy()
    pose = model_pose_results[0]

    w, h = frame.shape[1], frame.shape[0]
    keypoints = processKeypoints(
        pose.pred_instances.keypoints[0], w, h 
    )

    createTxtFileFromKeypoints(img_path, w, h, keypoints, output_dir=input_dir)

Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\rtmo\coco\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.pth


d:\magistrska\mmpose\mmpose\datasets\datasets\utils.py:102: UserWarning: The metainfo config file "configs/_base_/datasets/coco.py" does not exist. A matched config file "d:\magistrska\mmpose\mmpose\.mim\configs\_base_\datasets\coco.py" will be used instead.
  warnings.warn(


Loads checkpoint by local backend from path: D:\Magistrska\mmpose\checkpoints\yolox\yoloxpose_l_8xb32-300e_coco-640-de0f8dee_20230829.pth


d:\ProgramFiles\Anaconda\envs\openmmlab-gpu\lib\site-packages\mmdet\apis\inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(


Processing image (1/3135)
Processing image (2/3135)
Processing image (3/3135)
Processing image (4/3135)
Processing image (5/3135)
Processing image (6/3135)
Processing image (7/3135)
Processing image (8/3135)
Processing image (9/3135)
Processing image (10/3135)
Processing image (11/3135)
Processing image (12/3135)
Processing image (13/3135)
Processing image (14/3135)
Processing image (15/3135)
Processing image (16/3135)
Processing image (17/3135)
Processing image (18/3135)
Processing image (19/3135)
Processing image (20/3135)
Processing image (21/3135)
Processing image (22/3135)
Processing image (23/3135)
Processing image (24/3135)
Processing image (25/3135)
Processing image (26/3135)
Processing image (27/3135)
Processing image (28/3135)
Processing image (29/3135)
Processing image (30/3135)
Processing image (31/3135)
Processing image (32/3135)
Processing image (33/3135)
Processing image (34/3135)
Processing image (35/3135)
Processing image (36/3135)
Processing image (37/3135)
Processing

### Dataset Spliting

In [ ]:
dir = r"D:\Magistrska\blindoff-dataset\selected_images"

weights = (0.8, 0.1, 0.1)

Autosplitting images from D:\Magistrska\blindoff-dataset\selected_images
: 100% ━━━━━━━━━━━━ 3135/3135 1.3Kit/s 2.4s0.1s
